# Notebook16 actual compression and end-to-end speed gates

Inputs: frozen15learnedViT/geometry checkpoints ordeclared14fallback andonlyoriginaltrainingimages. Outputs: forward/backward errors includingactualQKV/projection/newgeometryparameters, observedadaptive ranks/work andsynchronizedCUDA time/peak at256/1024/4096tokens; completeJEPA256px/effective128timing. No N²farconstruction inhierarchicalpath; independentexactreferenceandoptionalSDPAbias mayallocateN²andarelabelled. Runtime typicallyminutes; hard12hourparentclockneverresets. Compressioncasescheckpointatomicallyincompression_progress.json andresumeonlymatching source/image/configidentities. Allscientificweightsdiscardedforcalibration.

In [ ]:
"""Measured training-only learned-token compression and full-JEPA calibration gates."""
from pathlib import Path
import copy,gc,json,math,time,statistics,types
import nbformat,torch
import torch.nn.functional as F
ROOT=Path('/home/benjamin/workspace/ahr_go/fmm_jeppa')
_p=ROOT/'experiments/16_hierarchical_confirmation/common.ipynb';COMMON=types.ModuleType('fmm16_common');COMMON.__file__=str(_p)
for _cell in nbformat.read(_p,4).cells:
    if _cell.cell_type=='code'and'ahr-export'in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_p),'exec'),COMMON.__dict__)


def _geometry(mixer,q):
    if hasattr(mixer,'geometry_parameters'):
        p=mixer.geometry_parameters(q.dtype);return p['temperature'],p['spatial_coefficient'],p['spatial_length']
    return None,None,None


def measured_operator(mixer,tokens,positions,valid,implementation,hierarchy,repeats=3,device='cuda:0'):
    """Same actual learned QKV/projection and dtype for dense, SDPA and hierarchical paths."""
    runner,encoder,op=COMMON.modules(ROOT);model=copy.deepcopy(mixer).to(device);model.requires_grad_(True)
    tokens=tokens.detach().to(device);positions=positions.to(device);valid=valid.to(device)
    names=list(dict(model.named_parameters()));parameters=tuple(model.parameters());observed=[];reference=None;operator_stats=None
    def forward(x):
        B,N,D=x.shape;safe=torch.where(valid[...,None],x,torch.zeros_like(x))
        q,k,v=model.qkv(safe).reshape(B,N,3,model.heads,D//model.heads).permute(2,0,3,1,4).unbind(0)
        tau,lam,ell=_geometry(model,q)
        if implementation=='hierarchical':
            values,stats=op.hierarchical_attention(q,k,v,valid,positions,tau,lam,ell,return_stats=True,**hierarchy)
        elif implementation=='sdpa':values=op.sdpa_attention(q,k,v,valid,positions,tau,lam,ell);stats={'algorithm':'torch.scaled_dot_product_attention actual runtime dispatch','dense_spatial_bias':lam is not None}
        else:values=op.exact_attention(q,k,v,valid,positions,tau,lam,ell);stats={'algorithm':'independent dense reference validation only'}
        out=model.projection(values.transpose(1,2).reshape_as(x))*valid[...,None]
        return out,stats
    for repetition in range(int(repeats)+1):
        x=tokens.clone().requires_grad_();gc.collect()
        if str(device).startswith('cuda'):
            torch.cuda.synchronize(device);torch.cuda.reset_peak_memory_stats(device)
        begin=time.perf_counter();out,stats=forward(x)
        if str(device).startswith('cuda'):torch.cuda.synchronize(device)
        middle=time.perf_counter();objective=out.square().mean()+out.mean()*.01
        gradients=torch.autograd.grad(objective,(x,)+parameters,allow_unused=True)
        if str(device).startswith('cuda'):torch.cuda.synchronize(device)
        if not torch.isfinite(out).all()or any(g is not None and not torch.isfinite(g).all()for g in gradients):raise FloatingPointError('Nonfinite observed operator output/gradient; promotion forbidden')
        end=time.perf_counter();peak=torch.cuda.max_memory_allocated(device)if str(device).startswith('cuda')else None
        reference={'output':out.detach().cpu(),'gradients':{'input':gradients[0].detach().cpu(),**{name:torch.zeros_like(param.detach().cpu())if grad is None else grad.detach().cpu()for name,param,grad in zip(names,parameters,gradients[1:])}}}
        if repetition:
            observed.append({'forward_seconds':middle-begin,'backward_seconds':end-middle,'forward_backward_seconds':end-begin,'peak_cuda_allocated_bytes':peak})
        operator_stats=stats;del x,out,objective,gradients
    result={'status':'MEASURED','precision':'float32'if tokens.dtype==torch.float32 else str(tokens.dtype),'implementation':implementation,
        'timing':'Synchronized actual forward and autograd backward, including adaptive topology/pivot controls and normalization; warmup excluded.',
        'forward_seconds':statistics.median(x['forward_seconds']for x in observed),'backward_seconds':statistics.median(x['backward_seconds']for x in observed),
        'forward_backward_seconds':statistics.median(x['forward_backward_seconds']for x in observed),
        'peak_cuda_allocated_bytes':max((x['peak_cuda_allocated_bytes']for x in observed if x['peak_cuda_allocated_bytes']is not None),default=None),
        'measurements':observed,'operator':operator_stats}
    del model;gc.collect()
    if str(device).startswith('cuda'):torch.cuda.empty_cache()
    return result,reference


def learned_tokens(model,bundle,config,ids,image_size,layer):
    from fmm_jepa.food101_data import EvaluationDataset
    changed=copy.deepcopy(config);changed['image_size']=int(image_size);data=EvaluationDataset(bundle['dataset'],ids,changed)
    images=torch.stack([data[i]['image']for i in range(len(ids))]);target=model.target.to(config['device']).eval();captured=[]
    handle=target.blocks[int(layer)].norm1.register_forward_hook(lambda module,args,out:captured.append(out.detach().clone()))
    try:
        with torch.no_grad():target(images.to(config['device']))
    finally:handle.remove()
    tokens=captured[-1].cpu();side=int(image_size)//int(config['model']['patch_size']);N=side*side
    indices=torch.arange(N).expand(len(ids),-1);positions=torch.stack(((indices//side+.5)/side,(indices%side+.5)/side),-1).float()
    return tokens,positions,torch.ones((len(ids),N),dtype=torch.bool),copy.deepcopy(target.blocks[int(layer)].mixer).cpu()


def _errors(reference,candidate):
    output=float((candidate['output']-reference['output']).norm()/reference['output'].norm().clamp_min(1e-12))
    errors={name:float((candidate['gradients'][name]-value).norm()/value.norm().clamp_min(1e-8))for name,value in reference['gradients'].items()}
    return {'relative_output_l2':output,'relative_gradient_l2':errors,'max_relative_gradient_l2':max(errors.values(),default=0.)}


def compression_benchmark(root,artifact,handoff,hierarchy=None,deadline=None,profile='full'):
    root,artifact=Path(root),Path(artifact);hierarchy=dict(hierarchy or{'leaf_size':32,'max_rank':16,'tolerance':1e-4,'eta':1.})
    runner,encoder,operator=COMMON.modules(root);device='cpu'if profile=='validation'else'cuda:0'
    cfg={'device':device};COMMON.environment(cfg);sources=COMMON.source_hashes(root)
    identity={'source_hashes':sources,'hierarchy':hierarchy,'token_counts':[256,1024,4096],'mask_policy':'Full valid source/query tokens; padding is separately software-tested.',
      'training_id_selection':'First and midpoint selectedtraining IDs, independently forfirst/lastencoderlayers; neverdevelopment/test','precision':'float32','handoff':handoff,'profile':profile}
    COMMON.immutable(artifact/'compression_registration.json',identity);path=artifact/'compression_progress.json'
    progress=COMMON.read(path)if path.exists()else{'status':'RUNNING','identity':identity,'completed':{}}
    if progress['identity']!=identity:raise ValueError('Compression resume identity differs')
    rows=[]
    if profile=='validation':
        torch.manual_seed(1616);mixer=torch.nn.Module();mixer.dim=8;mixer.heads=1;mixer.qkv=torch.nn.Linear(8,24);mixer.projection=torch.nn.Linear(8,8)
        cases=[('SOFTWARE_FIXTURE',None,32,torch.randn(1,32,8)*.01,torch.stack(torch.meshgrid(torch.arange(4),torch.arange(8),indexing='ij'),-1).reshape(1,32,2).float()/8,torch.ones(1,32,dtype=torch.bool),mixer,{'role':'SYNTHETIC_SOFTWARE_VALIDATION'})]
    else:
        cases=[]
        for variant in('vit','optional_geometry'):
            model,config,source=COMMON.learned_source(root,handoff,variant);config['device']=device;COMMON.environment(config)
            bundle=COMMON.data_bundle(root,artifact/('data_'+str(config['image_size'])),config,verify_images=True);ids=[bundle['train_ids'][0],bundle['train_ids'][len(bundle['train_ids'])//2]]
            for N in(256,1024,4096):
                size=int(math.sqrt(N))*int(config['model']['patch_size'])
                for layer in(0,len(model.target.blocks)-1):
                    for sample_id in ids:
                        COMMON.check_deadline(deadline,10)
                        key=variant+'-n'+str(N)+'-l'+str(layer)+'-'+str(ids.index(sample_id))
                        if key in progress['completed']:rows.append(progress['completed'][key]);continue
                        tokens,pos,valid,mixer=learned_tokens(model,bundle,config,[sample_id],size,layer)
                        cases.append((key,sample_id,N,tokens,pos,valid,mixer,source))
            model.cpu();del model;gc.collect();torch.cuda.empty_cache()
    for key,sample_id,N,tokens,pos,valid,mixer,source in cases:
        COMMON.check_deadline(deadline,5)
        measured={};reference=None;errors={}
        for implementation in('dense','sdpa','hierarchical'):
            value,payload=measured_operator(mixer,tokens,pos,valid,implementation,hierarchy,repeats=1 if profile=='validation'else 3,device=device)
            measured[implementation]=value
            if implementation=='dense':reference=payload
            else:errors[implementation]=_errors(reference,payload)
        row={'key':key,'tokens':N,'sample_id':sample_id,'source':source,'measurements':measured,'errors':errors,
          'hierarchical_speed_over_sdpa':measured['sdpa']['forward_backward_seconds']/measured['hierarchical']['forward_backward_seconds'],
          'operator_error_pass':errors['hierarchical']['relative_output_l2']<=1e-3 and errors['hierarchical']['max_relative_gradient_l2']<=1e-2,
          'evidence_role':'SYNTHETIC_SOFTWARE_VALIDATION'if profile=='validation'else'ACTUAL_FROZEN_TRAINING_TOKENS'}
        rows.append(row);progress['completed'][key]=row;COMMON.atomic_json(path,progress)
    error_pass=bool(rows)and all(r['operator_error_pass']for r in rows)
    large=[r for r in rows if r['tokens']>=1024];speed_pass=bool(large)and all(r['hierarchical_speed_over_sdpa']>=1.1 for r in large)
    work_pass=bool(large)and all(r['measurements']['hierarchical']['operator']['entry_work_over_dense']<.8 for r in large)
    report={'status':'COMPLETE','source_hashes':sources,'hierarchy':hierarchy,'rows':rows,'error_gate':error_pass,'large_token_speed_gate':speed_pass,
      'sampled_work_gate':work_pass,'profile':profile,'confirmation_eligible':profile=='full'and error_pass and speed_pass and work_pass,
      'no_uniform_kernel_error_certificate':True,'no_global_approximate_entry_positivity_guarantee':True,'gradient_scope':'Piecewise derivative through selected differentiable factors; detached discrete ACA pivots and topology. Full independently measured learned-token output and all parameter gradients audited.','scope':'Frozen learned training-only tokens atactual256/1024/4096; first/lastlayers andtwo predeclaredtrainingimages. Conditionalcompression evidence, notimagequality ormanifoldproof. Dense matrices exist onlyin independentreference/SDPAspatialbias, neverhierarchicalfarconstruction.'}
    progress['status']='COMPLETE';COMMON.atomic_json(path,progress);COMMON.atomic_json(artifact/'compression_benchmark.json',report);return report


def full_jepa_speed_gate(root,artifact,handoff,hierarchy,deadline=None,image_size=256,microbatch=32,repetitions=3):
    """Actual training data, effectivebatch128 and complete optimizer/EMA steps, discarded weights."""
    root,artifact=Path(root),Path(artifact);runner,encoder,operator=COMMON.modules(root)
    model,config,source=COMMON.learned_source(root,handoff,'optional_geometry');model.cpu()
    config=copy.deepcopy(config);config.update(device='cuda:0',image_size=int(image_size))
    config['training'].update(microbatch=int(microbatch),accumulation_steps=128//int(microbatch),updates=4000,checkpoint_every=50)
    if 128%int(microbatch):raise ValueError('Microbatch must divide128')
    COMMON.environment(config);bundle=COMMON.data_bundle(root,artifact/('data_'+str(config['image_size'])),config,verify_images=True)
    from fmm_jepa.food101_data import make_batch
    from fmm_jepa.semantic_pilot import pretraining_forward,move_masks,make_optimizer
    identities=bundle['train_ids'][:128];results={}
    for mode in('optimized_vit','exact_geometry','hierarchical'):
        COMMON.check_deadline(deadline,30)
        if mode=='optimized_vit':
            cfg=copy.deepcopy(config);cfg['kernel15']['variant']='vit';current=runner._default_build_model(cfg,'vit')
            source_state=model.state_dict();own=current.state_dict();current.load_state_dict({k:source_state[k]for k in own})
        else:current=copy.deepcopy(model)
        if mode=='hierarchical':operator.replace_encoder_mixers(current,hierarchy)
        current.to(config['device']);policy=runner._strict_training_protocol(config);optimizer,scheduler,groups=make_optimizer(current,policy)
        measurements=[]
        for repeat in range(int(repetitions)+1):
            COMMON.check_deadline(deadline,10);torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();started=time.perf_counter();optimizer.zero_grad(set_to_none=True);data_wall=0.;events=[]
            for micro in range(config['training']['accumulation_steps']):
                data_started=time.perf_counter();a=micro*microbatch;b=(micro+1)*microbatch;batch=make_batch(bundle['dataset'],identities[a:b],[repeat]*microbatch,config,training=True)
                images=batch['images'].to(config['device']);masks=move_masks(batch['masks'],torch.device(config['device']));data_wall+=time.perf_counter()-data_started
                ev0=torch.cuda.Event(enable_timing=True);ev1=torch.cuda.Event(enable_timing=True);ev0.record()
                output=pretraining_forward(current,{'images':images,'masks':masks});(output['loss']/config['training']['accumulation_steps']).backward();ev1.record();events.append((ev0,ev1))
            grads=[p.grad for p in current.parameters()if p.requires_grad and p.grad is not None]
            if not grads or not all(bool(torch.isfinite(g).all())for g in grads):raise FloatingPointError('Nonfinite timing gradients')
            if any(p.grad is not None for p in current.target.parameters()):raise AssertionError('Timing teacher received gradients')
            update0=torch.cuda.Event(enable_timing=True);update1=torch.cuda.Event(enable_timing=True);update0.record()
            torch.nn.utils.clip_grad_norm_([p for p in current.parameters()if p.requires_grad],float(policy['gradient_clip']),error_if_nonfinite=True)
            optimizer.step();current.update_teacher(scheduler.ema_for(repeat+1));scheduler.step();update1.record();events.append((update0,update1));torch.cuda.synchronize();wall=time.perf_counter()-started;peak=torch.cuda.max_memory_allocated()
            if repeat:measurements.append({'complete_step_seconds':wall,'data_seconds':data_wall,'compute_cuda_seconds':sum(e0.elapsed_time(e1)/1000 for e0,e1 in events),'peak_cuda_allocated_bytes':peak,'finite_loss':bool(torch.isfinite(output['loss']))})
        results[mode]={'median_complete_step_seconds':statistics.median(m['complete_step_seconds']for m in measurements),'peak_cuda_allocated_bytes':max(m['peak_cuda_allocated_bytes']for m in measurements),'measurements':measurements,'optimizer_groups':groups,'policy':policy,'median_compute_cuda_seconds':statistics.median(m['compute_cuda_seconds']for m in measurements),'median_data_seconds':statistics.median(m['data_seconds']for m in measurements)}
        del current,optimizer;gc.collect();torch.cuda.empty_cache()
    speed=results['optimized_vit']['median_complete_step_seconds']/results['hierarchical']['median_complete_step_seconds']
    gate={'status':'PASS'if speed>=1.1 else 'FAIL','measured_end_to_end_speed_over_optimized_vit':speed,'required_speed':1.1,'gate_uses':'Measured complete end-to-end step wall including real image decoding/augmentation/masks; compute-only diagnostic separate','image_size':image_size,
      'tokens':(image_size//config['model']['patch_size'])**2,'microbatch':microbatch,'effective_batch':128,'results':results,'source':source,
      'scope':'ActualcompleteJEPA preprocessing+transfer+student/predictor/teacher+backward+clipping+AdamW+EMA, sameprecision/masks/effectivebatch. Discardedtimingweights; noqualityselection. Warmup excluded.'}
    COMMON.atomic_json(artifact/'full_jepa_speed_gate.json',gate);return gate
